# Depth Ablation: 2D vs. 2D + Depth Representations

This experiment uses a corrected depth representation. An earlier implementation computed depth-related features directly from MediaPipe's image-space `z` coordinate, which produced numerically implausible values (see `movement_features.py` for the corrected calculation). This version uses `pose_world_landmarks` — MediaPipe's dedicated metric-3D output — instead.

This experiment evaluates whether depth-aware 3D movement features improve movement classification compared with a 2D-only representation.

The experiment uses the same participant/session dataset and evaluation protocol established in the previous modelling experiments. The only variable changed is the feature representation:

* **2D-only:** image-space movement features
* **2D + depth:** image-space features augmented with world-space depth features

Depth features are derived from MediaPipe `pose_world_landmarks`, rather than image-space landmark z-coordinates.

The purpose of this experiment is to determine whether the corrected depth representation provides measurable additional information for movement classification.

## Research Question

**Does adding depth-aware 3D movement features improve movement classification compared with a 2D-only representation?**

### Hypothesis

If depth contains useful information for distinguishing the movement classes, the 2D + depth representation should achieve higher balanced accuracy than the 2D-only representation under the same evaluation conditions.

If performance is unchanged or decreases, this provides evidence that the current depth features do not add useful discriminative information under the present recording conditions.

In [16]:
# Imports

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.preprocessing import StandardScaler

## Experimental Setup

The experiment uses the four recorded participant/session files already used in the previous experiments.

The evaluation protocol is kept fixed so that differences between the two representations can be attributed to the inclusion of depth features rather than to changes in the dataset split or classifier.

The comparison uses:

- the same participant/session dataset
- the same training and test trials
- the same temporal aggregation
- the same normalization procedure
- the same classifier
- the same evaluation metrics

Only the feature representation changes.

For each participant, Session 01 is used as the held-out test session and Session 02 as the training session. This creates a fixed cross-session evaluation in which the model is evaluated on a separate recording session from the one used for training.

The same test trials are used for both feature representations.

### Evaluation Protocol and Relation to Previous Experiments

Unlike the leave-one-participant-out evaluation used in the baseline and personalisation experiments, this depth ablation pools both participants during training and evaluates on their subsequent recording sessions.

Therefore, `05_depth_ablation.ipynb` measures **cross-session generalisation**, not generalisation to an unseen participant. The 0.940 balanced-accuracy result should therefore not be compared directly with the participant-independent results reported in the baseline and personalisation experiments.

The purpose of this experiment is instead to compare the 2D and 2D + depth representations under an identical fixed cross-session evaluation protocol.

In [17]:
# Load dataset

from pathlib import Path

DATA_DIR = Path("../data/movement")

DATASET_FILES = [
    "P01_S01.csv",
    "P01_S02.csv",
    "P02_S01.csv",
    "P02_S02.csv",
]

frames = []

for filename in DATASET_FILES:
    path = DATA_DIR / filename
    df = pd.read_csv(path)
    frames.append(df)

dataset = pd.concat(
    frames,
    ignore_index=True,
)

print("Rows:", len(dataset))
print("Columns:", len(dataset.columns))
print("Participants:", dataset["participant_id"].unique())
print("Sessions:", dataset["session_id"].unique())

Rows: 5555
Columns: 14
Participants: <StringArray>
['P01', 'P02']
Length: 2, dtype: str
Sessions: <StringArray>
['S01', 'S02']
Length: 2, dtype: str


In [18]:
FEATURES_2D = [
    "shoulder_width",
    "head_offset_x",
    "head_offset_y",
    "torso_dx",
    "torso_dy",
    "movement_speed",
]

DEPTH_FEATURES = [
    "head_offset_z",
    "torso_dz",
    "movement_speed_3d",
]

FEATURES_2D_DEPTH = FEATURES_2D + DEPTH_FEATURES

IDENTIFIER_COLUMNS = [
    "participant_id",
    "session_id",
    "trial_id",
    "movement_label",
]

print("2D features:")
print(FEATURES_2D)

print("\nDepth features:")
print(DEPTH_FEATURES)

print("\n2D + depth features:")
print(FEATURES_2D_DEPTH)

2D features:
['shoulder_width', 'head_offset_x', 'head_offset_y', 'torso_dx', 'torso_dy', 'movement_speed']

Depth features:
['head_offset_z', 'torso_dz', 'movement_speed_3d']

2D + depth features:
['shoulder_width', 'head_offset_x', 'head_offset_y', 'torso_dx', 'torso_dy', 'movement_speed', 'head_offset_z', 'torso_dz', 'movement_speed_3d']


## Feature Representation

The **2D representation** contains six image-space features:

* shoulder width
* head offset in x/y
* torso displacement in x/y
* 2D movement speed

The **depth-aware representation** adds three features computed from MediaPipe world landmarks:

* head offset in z
* torso displacement in z
* 3D movement speed

Thus:

**2D:** 6 features

**2D + depth:** 9 features

No other features are introduced between the two conditions.

In [19]:
# Trial-level aggregation

def aggregate_trials(df, feature_columns):
    aggregation = {}

    for feature in feature_columns:
        aggregation[feature] = ["mean", "std"]

    grouped = (
        df
        .groupby(
            [
                "participant_id",
                "session_id",
                "trial_id",
                "movement_label",
            ]
        )
        .agg(aggregation)
    )

    grouped.columns = [
        f"{feature}_{stat}"
        for feature, stat in grouped.columns
    ]

    grouped = grouped.reset_index()

    return grouped


trial_data_2d = aggregate_trials(
    dataset,
    FEATURES_2D,
)

trial_data_2d_depth = aggregate_trials(
    dataset,
    FEATURES_2D_DEPTH,
)

print("2D trial rows:", len(trial_data_2d))
print("2D + depth trial rows:", len(trial_data_2d_depth))

2D trial rows: 100
2D + depth trial rows: 100


In [20]:
trial_ids_2d = trial_data_2d[
    IDENTIFIER_COLUMNS
].copy()

trial_ids_2d_depth = trial_data_2d_depth[
    IDENTIFIER_COLUMNS
].copy()

assert trial_ids_2d.equals(
    trial_ids_2d_depth
)

print(
    "PASS - both representations contain "
    "the identical trial set."
)

PASS - both representations contain the identical trial set.


The 18-feature 2D + depth representation is compositionally equivalent to the **Reduced representation** used in the baseline and personalisation experiments: it contains the mean and standard deviation of the same nine underlying movement features.

The difference in this experiment is therefore not the underlying feature construction, but the explicit comparison between the six 2D features and the full nine-feature representation under the fixed cross-session depth ablation protocol.

## Fixed Cross-Session Evaluation

The held-out test set is defined once, before comparing the two feature representations.

For each participant and movement class:

- the first five trials, corresponding to Session 01, are assigned to the test set;
- the five trials from Session 02 form the training set.

Because each participant has two recording sessions with five repetitions of every movement, this produces:

- **Test set:** Session 01 — 25 trials per participant
- **Training set:** Session 02 — 25 trials per participant

Across both participants:

- **Training:** 50 trials
- **Test:** 50 trials

Therefore, the experiment evaluates cross-session generalisation: models learn from one recording session and are evaluated on a separate recording session from the same participant.

The same test trials are used for both representations.

In [21]:
TEST_TRIALS_PER_CLASS = 5

def create_fixed_split(trial_data):
    test_indices = []

    for (participant, movement), group in trial_data.groupby(
        ["participant_id", "movement_label"]
    ):
        group = group.sort_values(
            ["session_id", "trial_id"]
        )

        test_indices.extend(
            group.index[:TEST_TRIALS_PER_CLASS]
        )

    test_indices = set(test_indices)

    train_data = trial_data[
        ~trial_data.index.isin(test_indices)
    ].copy()

    test_data = trial_data[
        trial_data.index.isin(test_indices)
    ].copy()

    return train_data, test_data


train_2d, test_2d = create_fixed_split(
    trial_data_2d
)

train_2d_depth, test_2d_depth = create_fixed_split(
    trial_data_2d_depth
)

assert train_2d[IDENTIFIER_COLUMNS].equals(
    train_2d_depth[IDENTIFIER_COLUMNS]
)

assert test_2d[IDENTIFIER_COLUMNS].equals(
    test_2d_depth[IDENTIFIER_COLUMNS]
)

print("Training trials:", len(train_2d))
print("Test trials:", len(test_2d))
print()
print(
    "PASS - identical train/test split "
    "for both representations."
)

Training trials: 50
Test trials: 50

PASS - identical train/test split for both representations.


In [22]:
# Verify cross-session split

print("Training sessions:")
display(
    train_2d[
        ["participant_id", "session_id"]
    ]
    .drop_duplicates()
    .sort_values(
        ["participant_id", "session_id"]
    )
)

print("Test sessions:")
display(
    test_2d[
        ["participant_id", "session_id"]
    ]
    .drop_duplicates()
    .sort_values(
        ["participant_id", "session_id"]
    )
)

for participant in sorted(
    dataset["participant_id"].unique()
):
    train_sessions = set(
        train_2d.loc[
            train_2d["participant_id"] == participant,
            "session_id",
        ]
    )

    test_sessions = set(
        test_2d.loc[
            test_2d["participant_id"] == participant,
            "session_id",
        ]
    )

    assert train_sessions.isdisjoint(
        test_sessions
    )

print()
print(
    "PASS - training and test observations "
    "come from separate sessions for each participant."
)

Training sessions:


,participant_id,session_id
25,P01,S02
75,P02,S02


Test sessions:


,participant_id,session_id
0,P01,S01
50,P02,S01



PASS - training and test observations come from separate sessions for each participant.


In [23]:
def feature_columns_for(
    feature_columns,
    statistics=("mean", "std"),
):
    return [
        f"{feature}_{stat}"
        for feature in feature_columns
        for stat in statistics
    ]


MODEL_FEATURES_2D = feature_columns_for(
    FEATURES_2D
)

MODEL_FEATURES_2D_DEPTH = feature_columns_for(
    FEATURES_2D_DEPTH
)

X_train_2d = train_2d[
    MODEL_FEATURES_2D
]

X_test_2d = test_2d[
    MODEL_FEATURES_2D
]

X_train_2d_depth = train_2d_depth[
    MODEL_FEATURES_2D_DEPTH
]

X_test_2d_depth = test_2d_depth[
    MODEL_FEATURES_2D_DEPTH
]

y_train = train_2d[
    "movement_label"
]

y_test = test_2d[
    "movement_label"
]

assert len(X_train_2d) == len(X_train_2d_depth)
assert len(X_test_2d) == len(X_test_2d_depth)

assert y_train.equals(
    train_2d_depth["movement_label"]
)

assert y_test.equals(
    test_2d_depth["movement_label"]
)

print("2D features:", X_train_2d.shape[1])
print("2D + depth features:", X_train_2d_depth.shape[1])

2D features: 12
2D + depth features: 18


In [24]:
# Split summary

split_summary = (
    test_2d
    .groupby(
        ["participant_id", "session_id"]
    )
    .size()
    .reset_index(
        name="test_trials"
    )
)

display(split_summary)

print(
    "Total training trials:",
    len(train_2d),
)

print(
    "Total test trials:",
    len(test_2d),
)

,participant_id,session_id,test_trials
0,P01,S01,25
1,P02,S01,25


Total training trials: 50
Total test trials: 50


In [25]:
scaler_2d = StandardScaler()

X_train_2d_scaled = scaler_2d.fit_transform(
    X_train_2d
)

X_test_2d_scaled = scaler_2d.transform(
    X_test_2d
)


scaler_2d_depth = StandardScaler()

X_train_2d_depth_scaled = (
    scaler_2d_depth.fit_transform(
        X_train_2d_depth
    )
)

X_test_2d_depth_scaled = (
    scaler_2d_depth.transform(
        X_test_2d_depth
    )
)

print(
    "PASS - normalization fitted on training "
    "data only."
)

PASS - normalization fitted on training data only.


In [26]:
model_2d = LogisticRegression(
    max_iter=2000,
    random_state=42,
)

model_2d_depth = LogisticRegression(
    max_iter=2000,
    random_state=42,
)

model_2d.fit(
    X_train_2d_scaled,
    y_train,
)

model_2d_depth.fit(
    X_train_2d_depth_scaled,
    y_train,
)

pred_2d = model_2d.predict(
    X_test_2d_scaled
)

pred_2d_depth = model_2d_depth.predict(
    X_test_2d_depth_scaled
)

In [27]:
results = pd.DataFrame(
    {
        "representation": [
            "2D",
            "2D + depth",
        ],
        "accuracy": [
            accuracy_score(
                y_test,
                pred_2d,
            ),
            accuracy_score(
                y_test,
                pred_2d_depth,
            ),
        ],
        "balanced_accuracy": [
            balanced_accuracy_score(
                y_test,
                pred_2d,
            ),
            balanced_accuracy_score(
                y_test,
                pred_2d_depth,
            ),
        ],
        "macro_f1": [
            f1_score(
                y_test,
                pred_2d,
                average="macro",
            ),
            f1_score(
                y_test,
                pred_2d_depth,
                average="macro",
            ),
        ],
    }
)

display(
    results.round(3)
)

,representation,accuracy,balanced_accuracy,macro_f1
0,2D,0.94,0.94,0.941
1,2D + depth,0.94,0.94,0.940


In [28]:
comparison = (
    results
    .set_index("representation")
    [
        [
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
        ]
    ]
)

print("Representation comparison")
display(
    comparison.round(3)
)

balanced_accuracy_difference = (
    comparison.loc[
        "2D + depth",
        "balanced_accuracy",
    ]
    -
    comparison.loc[
        "2D",
        "balanced_accuracy",
    ]
)

macro_f1_difference = (
    comparison.loc[
        "2D + depth",
        "macro_f1",
    ]
    -
    comparison.loc[
        "2D",
        "macro_f1",
    ]
)

print(
    "Balanced accuracy difference "
    f"(2D + depth − 2D): "
    f"{balanced_accuracy_difference:+.3f}"
)

print(
    "Macro F1 difference "
    f"(2D + depth − 2D): "
    f"{macro_f1_difference:+.3f}"
)

Representation comparison


,accuracy,balanced_accuracy,macro_f1
representation,,,
2D,0.94,0.94,0.941
2D + depth,0.94,0.94,0.940


Balanced accuracy difference (2D + depth − 2D): +0.000
Macro F1 difference (2D + depth − 2D): -0.001


In [29]:
print("=== 2D representation ===")
print(
    classification_report(
        y_test,
        pred_2d,
        zero_division=0,
    )
)

print("\n=== 2D + depth representation ===")
print(
    classification_report(
        y_test,
        pred_2d_depth,
        zero_division=0,
    )
)

=== 2D representation ===
              precision    recall  f1-score   support

lean_forward       0.83      1.00      0.91        10
   lean_left       1.00      1.00      1.00        10
  lean_right       1.00      0.90      0.95        10
     neutral       1.00      0.90      0.95        10
  raise_arms       0.90      0.90      0.90        10

    accuracy                           0.94        50
   macro avg       0.95      0.94      0.94        50
weighted avg       0.95      0.94      0.94        50


=== 2D + depth representation ===
              precision    recall  f1-score   support

lean_forward       0.91      1.00      0.95        10
   lean_left       1.00      1.00      1.00        10
  lean_right       1.00      0.90      0.95        10
     neutral       1.00      0.80      0.89        10
  raise_arms       0.83      1.00      0.91        10

    accuracy                           0.94        50
   macro avg       0.95      0.94      0.94        50
weighted avg    

In [30]:
labels = sorted(
    y_test.unique()
)

cm_2d = confusion_matrix(
    y_test,
    pred_2d,
    labels=labels,
)

cm_2d_depth = confusion_matrix(
    y_test,
    pred_2d_depth,
    labels=labels,
)

print("=== 2D confusion matrix ===")
print(
    pd.DataFrame(
        cm_2d,
        index=labels,
        columns=labels,
    )
)

print("\n=== 2D + depth confusion matrix ===")
print(
    pd.DataFrame(
        cm_2d_depth,
        index=labels,
        columns=labels,
    )
)

=== 2D confusion matrix ===
              lean_forward  lean_left  lean_right  neutral  raise_arms
lean_forward            10          0           0        0           0
lean_left                0         10           0        0           0
lean_right               1          0           9        0           0
neutral                  0          0           0        9           1
raise_arms               1          0           0        0           9

=== 2D + depth confusion matrix ===
              lean_forward  lean_left  lean_right  neutral  raise_arms
lean_forward            10          0           0        0           0
lean_left                0         10           0        0           0
lean_right               1          0           9        0           0
neutral                  0          0           0        8           2
raise_arms               0          0           0        0          10


## Result

The 2D representation achieved a balanced accuracy of **0.940** on the fixed cross-session test set.

The 2D + depth representation also achieved a balanced accuracy of **0.940**.

The difference in balanced accuracy was therefore **0.000 points**.

Macro F1 was also effectively unchanged between the two representations.

## Interpretation

Adding the three depth-aware features did not improve classification performance under the evaluation conditions used in this experiment.

The 2D representation already achieved strong performance on the held-out recording sessions, and adding `head_offset_z`, `torso_dz`, and `movement_speed_3d` produced no measurable improvement in balanced accuracy.

The confusion matrices show that the two representations make broadly similar predictions. The differences at the class level are small and do not translate into an overall performance gain for the depth-aware representation.

Therefore, within the current dataset and feature engineering approach, there is **no evidence that the added world-space depth features provide additional predictive value for movement classification**.

This does not imply that depth is generally uninformative for embodied movement recognition. Rather, it indicates that the specific depth representation evaluated here did not provide additional discriminative information under the present recording conditions.

## Conclusion

The depth ablation compared a 2D movement representation with the same representation augmented by three world-space depth features.

Under the fixed cross-session evaluation protocol, both representations achieved a balanced accuracy of **0.940** and an accuracy of **0.940**.

The addition of depth-aware features therefore **did not improve classification performance** in the current experiment.

The result provides **no evidence for an additional contribution from the current depth-aware features** under the tested recording conditions.

Given the identical performance and the additional feature complexity, the simpler 2D representation is sufficient for the current movement-classification task.

The finding is interpreted together with the baseline, feature-ablation, and subsequent personalization experiments rather than as a general statement about the usefulness of depth for embodied AI.